In [8]:
import duckdb
import pandas as pd
import polars as pl
import numpy as np
from deltalake import write_deltalake


con = duckdb.connect()

df_size = 10
loops = 101
delta_table_path = "./test_table"

# tried different options, but the error persists
df_create_class = pd.DataFrame
# df_create_class = pl.DataFrame
df_return_method = "df"
# return_method = "pl"
mode = "append"
# mode = "overwrite"
partition_by = None
# partition_by = ["c"]

for i in range(loops):
    # create df
    df = df_create_class({
        'a': np.random.randint(0, 5, size=df_size),
        'b': np.random.randint(0, 5, size=df_size),
        'c': np.random.randint(0, 5, size=df_size),
    })

    # write it
    if df_create_class == pd.DataFrame:
        write_deltalake(delta_table_path, df, mode="append", partition_by=partition_by)
    else:
        df.write_delta(delta_table_path, mode=mode, delta_write_options={"partition_by": partition_by})

    print("writes: ", i+1)

    # attempt to read back
    qry = f"""
    SELECT *
    FROM delta_scan('{delta_table_path}')
    """
    result_df = getattr(con.query(qry), df_return_method)()

    print('table size: ', len(result_df))


writes:  1


InvalidInputException: Invalid Input Error: Attempting to execute an unsuccessful or closed pending query result
Error: IO Error: Hit DeltaKernel FFI error (from: While trying to read from delta table: 'file:///Users/tyler/Documents/repos/binance-public-data/python/rsch/notebooks/test_table/'): Hit error: 2 (ArrowError) with message (Invalid argument error: Incorrect datatype for StructArray field "partitionValues", expected Map(Field { name: "entries", data_type: Struct([Field { name: "keys", data_type: Utf8, nullable: false, dict_id: 0, dict_is_ordered: false, metadata: {} }, Field { name: "values", data_type: Utf8, nullable: false, dict_id: 0, dict_is_ordered: false, metadata: {} }]), nullable: true, dict_id: 0, dict_is_ordered: false, metadata: {} }, false) got Map(Field { name: "entries", data_type: Struct([Field { name: "key", data_type: Utf8, nullable: false, dict_id: 0, dict_is_ordered: false, metadata: {} }, Field { name: "value", data_type: Utf8, nullable: true, dict_id: 0, d

In [7]:
con.sql('FORCE install delta from core_nightly;')
# con.sql('UPDATE EXTENSIONS;')


In [11]:
df_create_class == pd.DataFrame

True

In [13]:
df_create_class = pl.DataFrame

df_create_class == pl.DataFrame

True

In [19]:
df

,a,b,c
0,3,1,1
1,3,1,1
2,2,4,0
3,3,4,1
4,3,3,1


In [8]:
import duckdb

duckdb.__version__

'1.0.0'

In [10]:
result_df

,id,part,value
0,5,1,value-5
1,7,1,value-7
2,9,1,value-9
3,11,1,value-11
4,13,1,value-13
5,10,0,value-10
6,12,0,value-12
7,14,0,value-14
8,6,0,value-6
9,8,0,value-8


In [1]:
2

2